In [1]:
"""
HomeGuard Security System — Programming Foundations Lab

Author: Armin Seyfi
Date: 2026-10-06

Simulate household sensors and generate security, safety, and comfort
alerts based on readings and the selected house mode.
Notifications are simulated by printing logs.
"""

import random as rd
import datetime as dt
import time

In [2]:
class Sensor:
    """Represent a household sensor with simulated readings."""

    def __init__(self, sensor_id, location, sensor_type):
        """Create a sensor with no initial reading."""
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.current_value = None

    def read(self):
        """Generate, store, and return a new reading."""
        if self.type == "temperature":
            self.current_value = rd.randint(30, 100)
        else:
            self.current_value = rd.choice([True, False])

        return self.current_value

    def isAbnormal(self):
        """Check a simple rule without considering house mode.

        Temperature is abnormal outside 65–75°F.
        Motion, door, and smoke are flagged when True.
        Alert rules decide whether a flagged reading needs an alert.
        """
        if self.current_value is None:
            return False

        if self.type == "temperature":
            return not 65 <= self.current_value <= 75

        return self.current_value is True

    def reset(self):
        """Clear the stored reading."""
        self.current_value = None

In [3]:
def sensor_list():
    """Create the four house sensor objects."""
    return [
        Sensor(1, "Living Room", "motion"),
        Sensor(2, "Front Door", "door"),
        Sensor(3, "Kitchen", "temperature"),
        Sensor(4, "Bedroom", "smoke"),
    ]


def update_sensor_values(sensors):
    """Generate an independent reading for each sensor."""
    for sensor in sensors:
        sensor.read()

In [4]:
def choose_mode():
    """Return HOME, AWAY, or SLEEP based on user input."""
    while True:
        choice = input(
            "Choose a mode: H = HOME, A = AWAY, S = SLEEP: "
        ).strip().upper()

        if choice == "H":
            return "HOME"
        elif choice == "A":
            return "AWAY"
        elif choice == "S":
            return "SLEEP"
        else:
            print("Invalid choice. Please enter H, A, or S.")

In [5]:
def create_alert_options():
    """Return allowed alert types and severity levels."""
    return {
        "security": ["Critical", "High", "Medium", "Low", None],
        "safety": ["Critical", "High", "Medium", "Low", None],
        "comfort": ["Critical", "High", "Medium", "Low", None],
    }

In [6]:
def check_alerts(sensors, current_mode, door_open_seconds):
    """Return triggered alerts based on readings, mode, and door duration."""
    alerts = []

    # Find sensors by type so their list order does not matter.
    by_type = {sensor.type: sensor for sensor in sensors}

    motion = by_type["motion"].current_value
    door_open = by_type["door"].current_value
    temperature = by_type["temperature"].current_value
    smoke = by_type["smoke"].current_value

    # SECURITY: individual alerts depend on house mode.
    if current_mode == "AWAY":
        if motion:
            alerts.append({
                "type": "security",
                "severity": "High",
                "message": "Motion detected while in AWAY mode!",
            })

        if door_open:
            alerts.append({
                "type": "security",
                "severity": "High",
                "message": "Front Door is open while in AWAY mode!",
            })

    elif current_mode == "SLEEP":
        if motion or door_open:
            alerts.append({
                "type": "security",
                "severity": "Medium",
                "message": "Motion detected or Front Door open during SLEEP mode.",
            })

    # Multiple security sensors active in the same round.
    if motion and door_open and current_mode in {"AWAY", "SLEEP"}:
        alerts.append({
            "type": "security",
            "severity": "High",
            "message": "Motion and open door detected together: possible break-in!",
        })

    # SAFETY: applies in every mode.
    if temperature is not None:
        if temperature < 35:
            alerts.append({
                "type": "safety",
                "severity": "High",
                "message": "Temperature below 35°F: frozen pipe risk!",
            })
        elif temperature > 95:
            alerts.append({
                "type": "safety",
                "severity": "High",
                "message": "Temperature above 95°F: possible equipment failure!",
            })

    if smoke:
        alerts.append({
            "type": "safety",
            "severity": "Critical",
            "message": "Smoke detected: fire risk!",
        })

    # COMFORT: applies only in HOME mode.
    if current_mode == "HOME":
        if temperature is not None and by_type["temperature"].isAbnormal():
            alerts.append({
                "type": "comfort",
                "severity": "Low",
                "message": "Temperature outside the comfort range of 65–75°F.",
            })

        if door_open and door_open_seconds > 300:
            alerts.append({
                "type": "comfort",
                "severity": "Low",
                "message": "Front Door has been observed open for over 5 minutes.",
            })

    return alerts

In [7]:
def create_alert_options():
    """Return the available alert types and severity levels.

    None means that no severity has been assigned.
    """
    return {
        "security": ["Critical", "High", "Medium", "Low", None],
        "safety": ["Critical", "High", "Medium", "Low", None],
        "comfort": ["Critical", "High", "Medium", "Low", None],
    }


alert_options = create_alert_options()

In [8]:
def check_alerts(sensors, current_mode, door_open_seconds):
    """Return triggered alerts based on readings, mode, and door duration."""
    alerts = []

    # Find sensors by type so their list order does not matter.
    by_type = {sensor.type: sensor for sensor in sensors}

    motion = by_type["motion"].current_value
    door_open = by_type["door"].current_value
    temperature = by_type["temperature"].current_value
    smoke = by_type["smoke"].current_value

    # SECURITY: individual alerts depend on house mode.
    if current_mode == "AWAY":
        if motion:
            alerts.append({
                "type": "security",
                "severity": "High",
                "message": "Motion detected while in AWAY mode!",
            })

        if door_open:
            alerts.append({
                "type": "security",
                "severity": "High",
                "message": "Front Door is open while in AWAY mode!",
            })

    elif current_mode == "SLEEP":
        if motion or door_open:
            alerts.append({
                "type": "security",
                "severity": "Medium",
                "message": "Motion detected or Front Door open during SLEEP mode.",
            })

    # Multiple security sensors active in the same round.
    if motion and door_open and current_mode in {"AWAY", "SLEEP"}:
        alerts.append({
            "type": "security",
            "severity": "High",
            "message": "Motion and open door detected together: possible break-in!",
        })

    # SAFETY: applies in every mode.
    if temperature is not None:
        if temperature < 35:
            alerts.append({
                "type": "safety",
                "severity": "High",
                "message": "Temperature below 35°F: frozen pipe risk!",
            })
        elif temperature > 95:
            alerts.append({
                "type": "safety",
                "severity": "High",
                "message": "Temperature above 95°F: possible equipment failure!",
            })

    if smoke:
        alerts.append({
            "type": "safety",
            "severity": "Critical",
            "message": "Smoke detected: fire risk!",
        })

    # COMFORT: applies only in HOME mode.
    if current_mode == "HOME":
        if temperature is not None and by_type["temperature"].isAbnormal():
            alerts.append({
                "type": "comfort",
                "severity": "Low",
                "message": "Temperature outside the comfort range of 65–75°F.",
            })

        if door_open and door_open_seconds > 300:
            alerts.append({
                "type": "comfort",
                "severity": "Low",
                "message": "Front Door has been observed open for over 5 minutes.",
            })

    return alerts

In [9]:
def print_report(sensors, current_mode, alerts, previous_door):
    """Print readings, alerts, and simulated notification logs."""
    time_now = dt.datetime.now().strftime("%H:%M:%S")

    print(f"\nTime: {time_now}")
    print(f"Mode: {current_mode}\n")

    for sensor in sensors:
        value = sensor.current_value
        label = sensor.location

        if sensor.type != "door":
            label += f" {sensor.type.capitalize()}"

        if value is None:
            status = "No reading yet"

        elif sensor.type == "motion":
            status = "Activity detected" if value else "No activity"

        elif sensor.type == "door":
            if value and previous_door is False:
                status = "OPENED"
            else:
                status = "OPEN" if value else "CLOSED"

        elif sensor.type == "temperature":
            if value < 35:
                description = "Frozen pipe risk"
            elif value > 95:
                description = "Equipment failure risk"
            elif sensor.isAbnormal():
                description = "Outside comfort range"
            else:
                description = "Normal"

            status = f"{value}°F ({description})"

        elif sensor.type == "smoke":
            status = "DETECTED" if value else "CLEAR"

        print(f"[READING] {label}: {status}")

    icons = {
        "security": "🚨",
        "safety": "⚠️",
        "comfort": "💡",
    }

    for alert in alerts:
        if alert["severity"] == "Critical":
            icon = "🔥"
        else:
            icon = icons[alert["type"]]

        print(
            f'[ALERT!] {icon} {alert["severity"].upper()}: '
            f'{alert["type"].upper()}: {alert["message"]}'
        )
        print(
            f"[LOG] [{time_now}] Sending notification to homeowner... "
            "(simulation only)"
        )

    if not alerts:
        print("[STATUS] No alerts.")

In [10]:
def main():
    """Run the simulator until the user interrupts it."""
    print("=== HomeGuard Security System ===")

    try:
        current_mode = choose_mode()
        sensors = sensor_list()
        alert_options = create_alert_options()

        door_sensor = next(
            sensor for sensor in sensors if sensor.type == "door"
        )

        previous_door = None
        door_open_since = None

        while True:
            update_sensor_values(sensors)
            now = time.monotonic()

            # Track consecutive open observations.
            if door_sensor.current_value:
                if door_open_since is None:
                    door_open_since = now

                door_open_seconds = now - door_open_since
            else:
                door_open_since = None
                door_open_seconds = 0

            alerts = check_alerts(
                sensors,
                current_mode,
                door_open_seconds,
            )

            # Check that generated alerts use the defined options.
            for alert in alerts:
                if alert["severity"] not in alert_options[alert["type"]]:
                    raise ValueError("Invalid alert severity.")

            print_report(
                sensors,
                current_mode,
                alerts,
                previous_door,
            )

            previous_door = door_sensor.current_value

            time.sleep(5)  # Generate another report after five seconds.

    except KeyboardInterrupt:
        print("\nSimulation stopped.")


main()

=== HomeGuard Security System ===

Time: 20:29:55
Mode: HOME

[READING] Living Room Motion: Activity detected
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 57°F (Outside comfort range)
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected: fire risk!
[LOG] [20:29:55] Sending notification to homeowner... (simulation only)
[ALERT!] 💡 LOW: COMFORT: Temperature outside the comfort range of 65–75°F.
[LOG] [20:29:55] Sending notification to homeowner... (simulation only)

Time: 20:30:00
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 44°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR
[ALERT!] 💡 LOW: COMFORT: Temperature outside the comfort range of 65–75°F.
[LOG] [20:30:00] Sending notification to homeowner... (simulation only)

Time: 20:30:05
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[READING] Kitchen Temperature: 51°F (Outside comfort ran